# Drive Stats Quarantine — DDL

**Purpose:**  
Create the quarantine table used to store invalid Backblaze drive observations rejected during the Curated/Silver ETL process.

**Source:** `backblaze_data.ing_bronze.drive_observations`  
**Target:** `backblaze_data.sot_silver.drive_stats_quarantine`

## Responsibilities

- Define a stable schema for quarantined records.
- Preserve the original record payload for investigation and debugging.
- Record the source period and file path for traceability.
- Store the rejection reason, validation rule, and quarantine timestamp.
- Create the table only if it does not already exist.
- Do not load, overwrite, or delete quarantine records in this DDL notebook.

## Quarantine Rules

Records may be quarantined for reasons such as:

- Unparseable or missing dates.
- Missing required identity fields, such as `serial_number`.
- Invalid `failure` values.
- Duplicate records at the expected grain: `(_period, date, serial_number)`.

Valid dates in different supported formats must not be quarantined solely because their formatting differs.

Schema-evolution columns that are legitimately NULL must not automatically cause quarantine.

## Execution

Run this notebook before `nb_func_drive_daily_stats.ipynb`.

The functional ETL notebook is responsible for identifying invalid records and writing them to this table.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)

SOURCE_TABLE = "backblaze_data.ing_bronze.drive_observations"
TARGET_TABLE = "backblaze_data.sot_silver.drive_stats_quarantine"

if not spark.catalog.tableExists(SOURCE_TABLE):
    raise ValueError(f"Source table does not exist: {SOURCE_TABLE}")

print(f"Source table validated: {SOURCE_TABLE}")
print(f"Quarantine target: {TARGET_TABLE}")

In [0]:
quarantine_schema = StructType([
    StructField("record_payload", StringType(), nullable=False),
    StructField("_period", StringType(), nullable=False),
    StructField("_source_file_path", StringType(), nullable=True),
    StructField("_ingested_at", TimestampType(), nullable=True),
    StructField("rejection_reason", StringType(), nullable=False),
    StructField("rule_name", StringType(), nullable=False),
    StructField("quarantined_at", TimestampType(), nullable=False)
])

print("Quarantine schema defined:")
print(quarantine_schema.simpleString())

In [0]:
# Creating the table only if it does not already exist.
if not spark.catalog.tableExists(TARGET_TABLE):
    (
        spark.createDataFrame([], quarantine_schema)
        .write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"Created empty quarantine table: {TARGET_TABLE}")
else:
    print(f"Table already exists; left unchanged: {TARGET_TABLE}")

display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))

In [0]:
target_df = spark.table(TARGET_TABLE)

expected_columns = [
    "record_payload",
    "_period",
    "_source_file_path",
    "_ingested_at",
    "rejection_reason",
    "rule_name",
    "quarantined_at"
]

if target_df.columns != expected_columns:
    raise ValueError(
        f"Unexpected quarantine schema. "
        f"Expected: {expected_columns}; Actual: {target_df.columns}"
    )

if target_df.count() != 0:
    print("Existing quarantine records found; data has been preserved.")

print(f"DDL validation passed. Columns: {len(target_df.columns)}")
print(f"Current quarantine row count: {target_df.count()}")